# 04 - Baseline estatistico de controle

Este notebook documenta o baseline estatistico simples anterior a CNN.

O objetivo e criar uma regua de controle: verificar se estatisticas globais simples dos pixels JPEG ja carregam sinal preditivo para distinguir `Healthy` e `Hepatic_Steatosis`.

Este baseline nao e modelo final e nao deve ser interpretado como ferramenta diagnostica.


## tl;dr

- O baseline usa atributos simples de intensidade: media, desvio padrao, minimo e maximo.
- Ele evita variaveis de caminho, nome do arquivo, tamanho do arquivo, `slice_id` e `inferred_group_id` como features preditivas.
- O split continua sendo o split por `inferred_group_id` definido nas etapas anteriores.
- A avaliacao e feita por slice e por grupo, com probabilidades agregadas por media no grupo.
- Os resultados servem como controle para interpretar se a CNN realmente agrega valor.


## Contexto & Metodos

Implementacao existente:

- `src/liverct/models/statistical_baseline.py`
- `scripts/run_statistical_baseline.py`

Entrada esperada:

- `data/interim/image_quality_audit.csv`

Saidas esperadas:

- `reports/tables/statistical_baseline_metrics.csv`
- `reports/tables/dummy_most_frequent_slice_predictions.csv`
- `reports/tables/dummy_most_frequent_group_predictions.csv`
- `reports/tables/logistic_regression_slice_predictions.csv`
- `reports/tables/logistic_regression_group_predictions.csv`
- `reports/tables/statistical_baseline_summary.json`

### Key Assumptions

- Analise preditiva de controle, nao analise causal.
- Nao ha tratamento/intervencao nem grupo controle causal.
- Outcome: `label`, com `1 = Hepatic_Steatosis` e `0 = Healthy`.
- Split por grupo ja foi validado anteriormente.
- Features sao derivadas de pixels JPEG e nao representam HU clinico.


## Setup

Carregamos bibliotecas, caminhos e as features declaradas no modulo do baseline.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
REPORT_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURES = PROJECT_ROOT / "reports" / "figures"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 40)


def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"Arquivo nao encontrado: {path}")
        return None
    return pd.read_csv(path)


def read_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        print(f"Arquivo nao encontrado: {path}")
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

from liverct.models.statistical_baseline import FEATURE_COLUMNS


## Data

Carregamos auditoria, split e resultados ja salvos do baseline estatistico.


In [2]:
audit_df = read_csv_if_exists(DATA_INTERIM / "image_quality_audit.csv")
split_slices = read_csv_if_exists(DATA_INTERIM / "split_slices.csv")
metrics_df = read_csv_if_exists(REPORT_TABLES / "statistical_baseline_metrics.csv")
summary = read_json_if_exists(REPORT_TABLES / "statistical_baseline_summary.json")

print("Features do baseline:", FEATURE_COLUMNS)
if metrics_df is None:
    print("Para gerar resultados: python scripts/run_statistical_baseline.py")


Features do baseline: ['mean_intensity', 'std_intensity', 'min_intensity', 'max_intensity']


## Validacao de leakage antes da modelagem

Antes de interpretar qualquer metrica, confirmamos novamente que nenhum `inferred_group_id` aparece em mais de um split.


In [3]:
if split_slices is not None:
    group_split_counts = split_slices.groupby("inferred_group_id")["split"].nunique()
    leaking_groups = group_split_counts[group_split_counts > 1]
    print("Grupos em mais de um split:", len(leaking_groups))
    if len(leaking_groups) == 0:
        display(Markdown("**OK:** split por grupo preservado."))
    else:
        display(leaking_groups.head(20))


Grupos em mais de um split: 0


**OK:** split por grupo preservado.

## Features usadas e excluidas

Features usadas:

- `mean_intensity`
- `std_intensity`
- `min_intensity`
- `max_intensity`

Variaveis propositalmente excluidas como preditores:

- `file_size_bytes`
- `filename`
- `file_path`
- `inferred_group_id`
- `slice_id`
- `md5`

Essa decisao reduz o risco de o baseline usar identificadores, caminho de arquivo ou artefatos tecnicos explicitos.


In [4]:
if audit_df is not None:
    required_columns = FEATURE_COLUMNS + ["label", "split", "inferred_group_id"]
    availability = pd.DataFrame({
        "column": required_columns,
        "available": [column in audit_df.columns for column in required_columns],
        "missing_values": [audit_df[column].isna().sum() if column in audit_df.columns else None for column in required_columns],
    })
    display(availability)

    if "split" in audit_df.columns:
        display(audit_df.groupby("split")["inferred_group_id"].nunique().reset_index(name="n_groups"))


,column,available,missing_values
0,mean_intensity,True,0
1,std_intensity,True,0
2,min_intensity,True,0
3,max_intensity,True,0
4,label,True,0
5,split,True,0
6,inferred_group_id,True,0


,split,n_groups
0,test,35
1,train,157
2,val,33


## Execucao opcional do baseline

Por padrao, esta celula nao reexecuta o baseline para evitar sobrescrever resultados derivados sem necessidade. Para reexecutar localmente, altere `RUN_BASELINE` para `True`.


In [5]:
RUN_BASELINE = False

if RUN_BASELINE:
    if audit_df is None:
        raise FileNotFoundError("Execute antes: python scripts/audit_images.py")
    from liverct.models.statistical_baseline import run_statistical_baseline

    result = run_statistical_baseline(audit_df=audit_df, output_dir=REPORT_TABLES)
    metrics_df = result["metrics_df"]
    display(metrics_df)
else:
    print("Execucao pulada. Para rodar no terminal: python scripts/run_statistical_baseline.py")


Execucao pulada. Para rodar no terminal: python scripts/run_statistical_baseline.py


## Metricas por slice e por grupo

Metricas minimas esperadas: `balanced_accuracy`, `recall_sensitivity`, `specificity`, `roc_auc`, `average_precision`, `precision` e `f1`. A classe positiva e `Hepatic_Steatosis`.


In [6]:
if metrics_df is not None:
    selected_columns = [
        "model", "level", "split", "n", "balanced_accuracy", "precision",
        "recall_sensitivity", "specificity", "f1", "roc_auc", "average_precision",
        "tn", "fp", "fn", "tp",
    ]
    display(metrics_df[selected_columns].sort_values(["model", "level", "split"]))


,model,level,split,n,balanced_accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,average_precision,tn,fp,fn,tp
3,dummy_most_frequent,group,test,35,0.500000,0.657143,1.000000,0.000000,0.793103,0.500000,0.657143,0,12,0,23
4,dummy_most_frequent,group,train,157,0.500000,0.662420,1.000000,0.000000,0.796935,0.500000,0.662420,0,53,0,104
5,dummy_most_frequent,group,val,33,0.500000,0.666667,1.000000,0.000000,0.800000,0.500000,0.666667,0,11,0,22
0,dummy_most_frequent,slice,test,560,0.500000,0.535714,1.000000,0.000000,0.697674,0.500000,0.535714,0,260,0,300
1,dummy_most_frequent,slice,train,2459,0.500000,0.554697,1.000000,0.000000,0.713576,0.500000,0.554697,0,1095,0,1364
2,dummy_most_frequent,slice,val,538,0.500000,0.524164,1.000000,0.000000,0.687805,0.500000,0.524164,0,256,0,282
9,logistic_regression,group,test,35,0.789855,0.840000,0.913043,0.666667,0.875000,0.833333,0.905722,8,4,2,21
10,logistic_regression,group,train,157,0.681150,0.802198,0.701923,0.660377,0.748718,0.756350,0.841979,35,18,31,73
11,logistic_regression,group,val,33,0.886364,0.950000,0.863636,0.909091,0.904762,0.933884,0.969722,10,1,3,19
6,logistic_regression,slice,test,560,0.733974,0.741325,0.783333,0.684615,0.761750,0.854949,0.876795,178,82,65,235


## Comparacao com a CNN 2D

Esta etapa nao reexecuta treinamento da CNN. Quando a tabela comparativa existir, ela e carregada apenas para contextualizar o baseline estatistico como regua de controle.


In [7]:
comparison_path = REPORT_TABLES / "baseline_cnn_vs_statistical_baseline_test.csv"
comparison_df = read_csv_if_exists(comparison_path)

if comparison_df is not None:
    display(comparison_df)
else:
    cnn_metrics_path = REPORT_TABLES / "baseline_cnn_test_metrics.csv"
    cnn_metrics = read_csv_if_exists(cnn_metrics_path)
    if cnn_metrics is not None:
        display(cnn_metrics)


,reference_model,level,split,reference_balanced_accuracy,reference_recall_sensitivity,reference_specificity,reference_f1,reference_roc_auc,reference_average_precision,cnn_balanced_accuracy,cnn_recall_sensitivity,cnn_specificity,cnn_f1,cnn_roc_auc,cnn_average_precision,delta_balanced_accuracy,delta_recall_sensitivity,delta_specificity,delta_f1,delta_roc_auc,delta_average_precision,reference_source
0,logistic_regression,slice,test,0.7340,0.7833,0.6846,0.7618,0.8549,0.8768,0.815513,0.823333,0.807692,0.827471,0.922744,0.945106,0.081513,0.040033,0.123092,0.065671,0.067844,0.068306,docs/04_baseline_estatistico_controle.md
1,logistic_regression,group,test,0.7899,0.9130,0.6667,0.8750,0.8333,0.9057,0.807971,0.782609,0.833333,0.837209,0.920290,0.962406,0.018071,-0.130391,0.166633,-0.037791,0.086990,0.056706,docs/04_baseline_estatistico_controle.md


## Predicoes por grupo

A avaliacao por grupo usa a media das probabilidades dos slices do mesmo `inferred_group_id`. Isso e mais coerente com a regra de split por grupo.


In [8]:
group_predictions = read_csv_if_exists(REPORT_TABLES / "logistic_regression_group_predictions.csv")
if group_predictions is not None:
    display(group_predictions.head(10))
    display(
        group_predictions.groupby("split")
        .agg(n_groups=("inferred_group_id", "nunique"), mean_prob_positive=("prob_positive", "mean"))
        .round(4)
        .reset_index()
    )


,split,inferred_group_id,label,prob_positive,n_slices,slice_positive_rate,pred_label
0,test,106-img-00060,0,0.302290,21,0.000000,0
1,test,114-img-00064,0,0.588079,24,1.000000,1
2,test,117-img-00066,1,0.801500,13,1.000000,1
3,test,126-img-00075,1,0.762031,13,1.000000,1
4,test,13-img-00012,1,0.523778,12,0.500000,1
5,test,131-img-00080,1,0.902438,12,1.000000,1
6,test,133-img-00082,0,0.165991,24,0.000000,0
7,test,143-img-00092,0,0.632503,24,0.916667,1
8,test,149-img-00097,0,0.353057,24,0.000000,0
9,test,160-img-00003,1,0.503905,10,0.500000,1


,split,n_groups,mean_prob_positive
0,test,35,0.5525
1,train,157,0.5377
2,val,33,0.5624


## Results

O baseline estatistico responde uma pergunta de controle: quanto sinal preditivo existe em estatisticas globais simples dos pixels?

Se ele tiver desempenho relevante, a interpretacao da CNN deve ser cautelosa, pois parte do sinal pode vir de intensidade, contraste, textura global, compressao JPEG ou outro artefato tecnico do dataset.


## Takeaways

O baseline estatistico e uma regua de controle, nao um modelo final.

Ele deve acompanhar os experimentos de CNN porque ajuda a separar ganho real de representacao visual mais rica, sinal global simples ja presente nos pixels, possiveis atalhos tecnicos e diferencas entre avaliacao por slice e avaliacao por grupo.

A etapa continua sendo preditiva exploratoria. Nao ha inferencia causal nem validacao clinica.
